# Ford Mustang Mach-E 2026 — 360° Video Transcript → Pinecone
### Ingestion pipeline for voice/speech transcripts from training videos

**Source:** `Talk/combined_segments.json` — 814 speech segments from the 360° walkthrough video  
**Namespace:** `video_transcript` (same Pinecone index: `ford-mache-erg`)  
**Chunking:** Sliding window of 10 segments, step 8 (2-segment overlap)

Re-run this notebook for any new `*_segments.json` files placed in `Talk/`.

## 1. Install Dependencies

In [1]:
!pip install -q openai python-dotenv pinecone-client

zsh:1: command not found: pip


## 2. Imports & Setup

In [2]:
import os
import json
import uuid

from dotenv import load_dotenv
load_dotenv()  # reads OPENAI_API_KEY, PINECONE_API_KEY from .env

import openai
from pinecone import Pinecone, ServerlessSpec

print("Imports loaded.")

Imports loaded.


## 3. Configuration

In [3]:
# ── Pinecone ───────────────────────────────────────────────────────────────
PINECONE_INDEX_NAME = "ford-mache-erg"
PINECONE_CLOUD      = "aws"
PINECONE_REGION     = "us-east-1"
NAMESPACE           = "video_transcript"

# ── Source files ──────────────────────────────────────────────────────────
TALK_DIR = os.path.join(os.getcwd(), "Talk")

# ── Chunking parameters ───────────────────────────────────────────────────
WINDOW_SIZE  = 10   # consecutive segments per chunk
STEP_SIZE    = 8    # advance by 8 → 2-segment overlap between consecutive chunks

# ── Embedding ─────────────────────────────────────────────────────────────
EMBEDDING_MODEL = "text-embedding-3-small"
EMBED_BATCH     = 50  # segments per OpenAI embedding API call

# ── Clients ───────────────────────────────────────────────────────────────
openai_client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
pc            = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

print(f"Talk dir      : {TALK_DIR}")
print(f"Pinecone index: {PINECONE_INDEX_NAME}")
print(f"Namespace     : {NAMESPACE}")
print(f"Window/step   : {WINDOW_SIZE}/{STEP_SIZE} ({WINDOW_SIZE - STEP_SIZE} seg overlap)")
print("Configuration loaded.")

Talk dir      : /Users/18668005@nebraska.edu/Documents/RAG_Responder/Talk
Pinecone index: ford-mache-erg
Namespace     : video_transcript
Window/step   : 10/8 (2 seg overlap)
Configuration loaded.


## 4. Pinecone Index Setup

In [4]:
existing_indexes = [idx.name for idx in pc.list_indexes()]
if PINECONE_INDEX_NAME not in existing_indexes:
    print(f"Creating index '{PINECONE_INDEX_NAME}'...")
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
else:
    print(f"Using existing index '{PINECONE_INDEX_NAME}'")

index = pc.Index(PINECONE_INDEX_NAME)
print(index.describe_index_stats())

Using existing index 'ford-mache-erg'


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'erg_full': {'vector_count': 37},
                'rescue_sheet': {'vector_count': 4},
                'video_transcript': {'vector_count': 102}},
 'total_vector_count': 143,
 'vector_type': 'dense'}


## 5. Chunking Helper

Groups consecutive speech segments into overlapping windows.  
Each chunk concatenates `segment.text` fields and carries temporal metadata
(`start_time`, `end_time`, `start_hms`, `end_hms`) so the chatbot can reference
approximate video timestamps in its answers.

In [5]:
def sliding_window_chunks(segments, window=10, step=8):
    """
    Yield dicts, each representing one chunk:
        text        : concatenated speech text for this window
        start_time  : float seconds at which this window begins
        end_time    : float seconds at which this window ends
        start_hms   : HH:MM:SS string
        end_hms     : HH:MM:SS string
        segment_ids : "first_id-last_id" inclusive range
        chunk_index : sequential 0-based index
    """
    n = len(segments)
    chunk_idx = 0
    for start in range(0, n, step):
        window_segs = segments[start : start + window]
        if not window_segs:
            break
        text = " ".join(s["text"].strip() for s in window_segs if s["text"].strip())
        if not text:
            chunk_idx += 1
            continue
        yield {
            "text":        text,
            "start_time":  float(window_segs[0]["start"]),
            "end_time":    float(window_segs[-1]["end"]),
            "start_hms":   window_segs[0]["start_hms"],
            "end_hms":     window_segs[-1]["end_hms"],
            "segment_ids": f"{window_segs[0]['id']}-{window_segs[-1]['id']}",
            "chunk_index": chunk_idx,
        }
        chunk_idx += 1

print("Chunking helper defined.")

Chunking helper defined.


## 6. Embedding Helper

In [6]:
def embed_texts(texts):
    """
    Embed a list of strings using text-embedding-3-small.
    Returns a list of 1536-dim float vectors in the same order.
    """
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    return [item.embedding for item in response.data]

print("Embedding helper defined.")

Embedding helper defined.


## 7. Full Ingestion Loop

Processes every `*_segments.json` file found in `Talk/`.  
Safe to re-run — Pinecone upsert is idempotent (same vector IDs overwrite).

In [7]:
import glob as _glob

transcript_files = sorted(_glob.glob(os.path.join(TALK_DIR, "*_segments.json")))
if not transcript_files:
    print(f"No *_segments.json files found in {TALK_DIR}")
else:
    print(f"Found {len(transcript_files)} transcript file(s):")
    for f in transcript_files:
        print(f"  {os.path.basename(f)}")

Found 3 transcript file(s):
  VID_20250912_122900_00_010_012_segments.json
  VID_20250912_134205_00_013_014_segments.json
  combined_segments.json


In [8]:
total_upserted = 0

for transcript_path in transcript_files:
    filename = os.path.basename(transcript_path)
    print(f"\n{'─'*60}")
    print(f"📝  {filename}  →  namespace: {NAMESPACE}")

    with open(transcript_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    segments = data["segments"]
    print(f"  Loaded {len(segments)} segments")

    chunks = list(sliding_window_chunks(segments, window=WINDOW_SIZE, step=STEP_SIZE))
    print(f"  Created {len(chunks)} chunks (window={WINDOW_SIZE}, step={STEP_SIZE})")

    # ── Embed in batches ──────────────────────────────────────────────────
    vectors = []
    for batch_start in range(0, len(chunks), EMBED_BATCH):
        batch = chunks[batch_start : batch_start + EMBED_BATCH]
        texts = [c["text"] for c in batch]
        embeddings = embed_texts(texts)

        for chunk, embedding in zip(batch, embeddings):
            vector_id = (
                f"{os.path.splitext(filename)[0]}"
                f"_chunk{chunk['chunk_index']:04d}"
            )
            vectors.append({
                "id": vector_id,
                "values": embedding,
                "metadata": {
                    "source_doc":   filename,
                    "doc_type":     "video_transcript",
                    "namespace":    NAMESPACE,
                    "vehicle":      "Ford Mustang Mach-E 2026",
                    "start_time":   chunk["start_time"],
                    "end_time":     chunk["end_time"],
                    "start_hms":    chunk["start_hms"],
                    "end_hms":      chunk["end_hms"],
                    "segment_ids":  chunk["segment_ids"],
                    "chunk_index":  chunk["chunk_index"],
                    "text":         chunk["text"][:1000],  # store preview for debugging
                },
            })

        print(f"  Embedded batch {batch_start // EMBED_BATCH + 1} "
              f"({len(batch)} chunks)")

    # ── Upsert to Pinecone ────────────────────────────────────────────────
    UPSERT_BATCH = 100
    for i in range(0, len(vectors), UPSERT_BATCH):
        batch_vecs = vectors[i : i + UPSERT_BATCH]
        index.upsert(vectors=batch_vecs, namespace=NAMESPACE)
        print(f"  Upserted vectors {i}–{i + len(batch_vecs) - 1}")

    total_upserted += len(vectors)
    print(f"  ✅ Done — {len(vectors)} vectors upserted from '{filename}'")

print(f"\n{'─'*60}")
print(f"Total vectors upserted this run: {total_upserted}")


────────────────────────────────────────────────────────────
📝  VID_20250912_122900_00_010_012_segments.json  →  namespace: video_transcript
  Loaded 1862 segments
  Created 233 chunks (window=10, step=8)


  Embedded batch 1 (50 chunks)


  Embedded batch 2 (50 chunks)


  Embedded batch 3 (50 chunks)


  Embedded batch 4 (50 chunks)


...

  Embedded batch 1 (50 chunks)


  Embedded batch 2 (50 chunks)


  Embedded batch 3 (2 chunks)


  Upserted vectors 0–99


  Upserted vectors 100–101
  ✅ Done — 102 vectors upserted from 'combined_segments.json'

────────────────────────────────────────────────────────────
Total vectors upserted this run: 537


## 8. Verify — Index Stats

In [9]:
stats = index.describe_index_stats()
print("Index stats:")
print(f"  Total vectors : {stats['total_vector_count']}")
print("\nPer-namespace breakdown:")
for ns, ns_stats in stats.get('namespaces', {}).items():
    print(f"  {ns:30s} : {ns_stats['vector_count']} vectors")

Index stats:
  Total vectors : 578

Per-namespace breakdown:
  erg_full                       : 37 vectors
  rescue_sheet                   : 4 vectors
  video_transcript               : 537 vectors


## 9. Smoke Test — Similarity Search

In [10]:
def query_transcript(question, k=3):
    """Run a similarity search against the video_transcript namespace."""
    q_embedding = embed_texts([question])[0]
    results = index.query(
        vector=q_embedding,
        top_k=k,
        namespace=NAMESPACE,
        include_metadata=True,
    )
    print(f"\n🔍  '{question}'")
    for match in results["matches"]:
        meta = match["metadata"]
        print(f"  [{match['score']:.4f}] "
              f"{meta['start_hms']}–{meta['end_hms']} "
              f"(segs {meta['segment_ids']})")
        print(f"    {meta['text'][:200]}...")

query_transcript("how to disconnect the high voltage system")
query_transcript("battery fire thermal runaway")
query_transcript("lifting points stabilization")


🔍  'how to disconnect the high voltage system'
  [0.6734] 00:32:56–00:33:06 (segs 713-722)
    disconnects all the high voltage. The goal of this tool is to go back to that idea where all the voltage is isolated back into battery counting....
  [0.6389] 00:37:26–00:37:55 (segs 305-314)
    So cut your cable, get the 12 volt system disconnected at the back. Okay? There's my two methods that I pretty much preach to everybody across. If you do those two things, you should feel very, very c...
  [0.6279] 00:34:35–00:35:06 (segs 241-250)
    As an operator of the vehicle, I turn the car off. If I turn the car off, the 12 volt system's de-energized and I no longer let the high voltage out of its battery housing space. That make sense? So t...



🔍  'battery fire thermal runaway'
  [0.5299] 00:00:00–00:01:22 (segs 1-10)
    above the door and up. Okay. I like to show this that this is a picture of a battery from a Tesla S model. Okay. So when they put their batteries in, they're using literally small double A size lithiu...
  [0.4850] 00:06:07–00:06:56 (segs 73-82)
    I mean, our GM has implemented a tool that will discharge the battery before shipping. So you got to hook it up somewhere in the corner of the shop and let it sit for about four days. I think the issu...
  [0.4764] 00:01:12–00:02:21 (segs 9-18)
    longer. They've had more opportunity to kind of build strategies. And if you talk to anybody over in Europe, they're going to tell you that once you've got a fire in the battery area of this, your bes...



🔍  'lifting points stabilization'
  [0.3221] 00:20:36–00:21:05 (segs 329-338)
    Starting to see a bunch of vehicles now that actually have little gas shocks and when a collision is imminent as detected by, for example, the radar sensors and the cameras, they actually pop the hood...
  [0.3188] 00:25:18–00:25:25 (segs 865-874)
    maintenance right no this is just the energy to move the vehicle that's all I'm looking...
  [0.3123] 05:15:57–05:16:50 (segs 737-746)
    Here's just to look and make sure we all understand what we're looking at you've got airbags in blue This is really just srs. Control unit This indicates that you've got stored gas some sort of pressu...


---
## Notes

| Parameter | Value | Rationale |
|---|---|---|
| `WINDOW_SIZE` | 10 segments | ~30–90 seconds of speech; covers a coherent topic block |
| `STEP_SIZE` | 8 | 2-segment overlap prevents context loss at chunk boundaries |
| `EMBED_BATCH` | 50 | Stays well within OpenAI's 2048-input limit per call |
| Namespace | `video_transcript` | Isolated from `erg_full` / `rescue_sheet` for clean routing |
| Metadata `text` | truncated to 1000 chars | Stored for debugging; Pinecone metadata cap is 40 KB per vector |

To add more transcript files: drop any `*_segments.json` into `Talk/` and re-run this notebook.
Pinecone upsert is idempotent — existing vectors are overwritten by their ID, not duplicated.